# ♻️ Sistem Klasifikasi Sampah Cerdas (Viskom & Pengolahan Citra)
Notebook Google Colab ini digunakan untuk menguji dan menjalankan pipeline klasifikasi citra sampah menggunakan **Ekstraksi Fitur Citra (Warna HSV/LAB, Bentuk/Morfologi, Tekstur GLCM & Tepi)** serta model **K-Nearest Neighbor (KNN)**.

## 1. Clone Repository & Install Dependencies
Menarik repository dari GitHub dan menginstal pustaka yang diperlukan.

In [ ]:
# Clone repository dari GitHub jika belum ada
!git clone https://github.com/akbarlubis247/Projek-Viskom-n-citra.git
%cd Projek-Viskom-n-citra

# Install requirements
!pip install -r requirements.txt
!pip install pyngrok streamlit-localtunnel

print("Repository siap dan dependensi berhasil di-install!")

Cloning into 'Projek-Viskom-n-citra'...
remote: Enumerating objects: 2426, done.
remote: Counting objects: 100% (2426/2426), done.
remote: Compressing objects: 100% (2422/2422), done.
remote: Total 2426 (delta 4), reused 2421 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (2426/2426), 43.39 MiB | 24.71 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (2414/2414), done.
/content/Projek-Viskom-n-citra


## 2. Uji Coba Model Langsung di Colab (Upload Gambar)
Upload foto sampah dari laptop/HP Anda untuk melihat visualisasi segmentasi dan hasil prediksi kategori sampahnya langsung di cell output.

In [ ]:
import cv2
import joblib
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from feature_extraction import extract_all_features

# 1. Load Model dan Scaler
model = joblib.load("models/knn_model.pkl")
scaler = joblib.load("models/scaler.pkl")

print("Model KNN dan Scaler berhasil dimuat!")
print("Silakan pilih foto gambar sampah untuk di-upload:")

# 2. Dialog Upload File
uploaded = files.upload()

label_map = {
    "botol_plastik": "Botol Plastik 🧴",
    "botol_kaca": "Botol Kaca 🍾",
    "kaleng": "Kaleng / Logam 🥫",
    "kertas": "Kertas 📄",
    "kardus": "Kardus 📦"
}

for filename in uploaded.keys():
    img = cv2.imread(filename)
    if img is None:
        print(f"Gagal membaca file {filename}")
        continue

    # Ekstraksi Fitur
    features, img_resized, mask = extract_all_features(img)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    masked_obj = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # Prediksi Model
    features_scaled = scaler.transform([features])
    prediction = model.predict(features_scaled)[0]
    nama_prediksi = label_map.get(prediction, prediction.upper())

    # Tampilkan Visualisasi
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Citra Input (256x256)")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("2. Mask Kontur Objek")
    axes[1].axis("off")

    axes[2].imshow(masked_obj)
    axes[2].set_title("3. Objek Tersegmentasi")
    axes[2].axis("off")
    plt.show()

    print(f"===================================================")
    print(f"Hasil Prediksi: {nama_prediksi}")

    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(features_scaled)[0]
        print("Probabilitas per kelas:")
        for cls, prob in zip(model.classes_, probs):
            print(f" - {label_map.get(cls, cls)}: {prob * 100:.2f}%")
    print(f"===================================================\n")

## 3. (Opsional) Jalankan Web App Streamlit dari Google Colab
Jika ingin membuka tampilan Web App Streamlit interaktif langsung dari Google Colab menggunakan localtunnel.

In [ ]:
# Dapatkan IP publik untuk password localtunnel
!curl ipv4.icanhazip.com

# Jalankan Streamlit di background & buat tunnel publik
!streamlit run app.py & npx localtunnel --port 8501